# Variable de control

Nicole Getzabel Flores Godines

21/10/26

Librerías

In [2]:
import numpy as np
from scipy.stats import norm

Datos iniciales

In [1]:
S0 = 100
r = 0.05
sigma = 0.20
K = 100
T = 1

n = 252
M = 100

dt = T / n


Simulación de las 100 rutas

In [4]:
np.random.seed(123)

rutas = np.zeros((M, n + 1))
rutas[:, 0] = S0

for i in range(M):
    for j in range(1, n + 1):
        Z = np.random.normal(0, 1)
        rutas[i, j] = rutas[i, j-1] * np.exp(
            (r - 0.5 * sigma**2) * dt
            + sigma * np.sqrt(dt) * Z
        )

print(rutas)


[[100.          98.65328916  99.91261888 ...  96.54387885  98.84839741
   99.90573946]
 [100.          99.15568196  97.07582275 ...  84.16617461  83.8951547
   84.89358849]
 [100.         101.61326509 103.35229278 ... 138.82870451 135.66009548
  137.572277  ]
 ...
 [100.          98.95711491  99.33587147 ...  64.98928242  64.34774327
   62.93707902]
 [100.          99.84923142  97.94479118 ...  86.05858196  84.31248902
   82.82188167]
 [100.         100.29944451 100.52509392 ... 108.36349517 107.05744691
  104.86011139]]


Promedio aritmético y geométrico

In [ ]:
aritmetico = np.mean(rutas[:, 1:], axis=1)

geometrico = np.exp(
    np.mean(np.log(rutas[:, 1:]), axis=1)
)


Cálculo del pago de las opciones Call

In [12]:
call_aritmetica = np.maximum(aritmetico - K, 0)
call_geometrica = np.maximum(geometrico - K, 0)

pago_aritmetico = np.exp(-r*T) * call_aritmetica
pago_geometrico = np.exp(-r*T) * call_geometrica


print("Precio Call aritmética:", np.mean(pago_aritmetico))
print("Precio Call geométrica:", np.mean(pago_geometrico))

Precio Call aritmética: 6.93097019136804
Precio Call geométrica: 6.641810278132507


Precio teórico de la Call geométrica


In [14]:
sigma_A = sigma / np.sqrt(3)

b = r
b_A = 0.5 * (b + sigma_A**2 / 2)

d1 = (np.log(S0/K) + (b_A + sigma_A**2/2)*T) / (sigma_A*np.sqrt(T))
d2 = d1 - sigma_A*np.sqrt(T)

# Precio teorico de la Call geometrica
C_G = (S0*np.exp((b_A-r)*T)*norm.cdf(d1)
       - K*np.exp(-r*T)*norm.cdf(d2))

print("Volatilidad ajustada:", sigma_A)
print("Costo de acarreo ajustado:", b_A)
print("Precio teórico Call geométrica:", C_G)


Volatilidad ajustada: 0.11547005383792516
Costo de acarreo ajustado: 0.028333333333333335
Precio teórico Call geométrica: 5.942176274329363


Aplicación de la variable de control

In [15]:
covarianza = np.cov(pago_geometrico, pago_aritmetico)[0, 1]
varianza_x = np.var(pago_geometrico, ddof=1)

beta = covarianza / varianza_x

m = n
sigma_G = sigma * np.sqrt((m+1)*(2*m+1)/(6*m**2))
mu_G = (r - sigma**2/2) * (m+1)/(2*m) + sigma_G**2/2

d1_G = (np.log(S0/K) + (mu_G + sigma_G**2/2)*T) / (sigma_G*np.sqrt(T))
d2_G = d1_G - sigma_G*np.sqrt(T)

C_G = np.exp(-r*T) * (
    S0*np.exp(mu_G*T)*norm.cdf(d1_G)
    - K*norm.cdf(d2_G)
)

control = pago_aritmetico - beta * (pago_geometrico - C_G)

print("Coeficiente b:", beta)
print("Precio sin control:", np.mean(pago_aritmetico))
print("Precio con control:", np.mean(control))


Coeficiente b: 1.0439703804557383
Precio sin control: 6.93097019136804
Precio con control: 5.807343360484765


Conclusión

En este ejercicio simulamos 100 rutas para calcular el precio de una opción Call asiática utilizando el método de variable de control. El precio sin control fue de 6.93 y con control de 5.81. También observamos que la varianza disminuyó de 81.72 a 0.1139, logrando una reducción del 99.86%. Esto demuestra que el método de variable de control ayuda a obtener resultados más precisos y con menor variabilidad.